# **Modelos LSTM — Total mensual de viajeros**

En este notebook se modela con redes **LSTM** la serie del **total mensual de viajeros internacionales** que ingresan al país, que es la serie obligatoria del Laboratorio 1 y la que agrega todas las regiones, vías y fronteras. Se construyen varias configuraciones de LSTM, se tunean sus hiperparámetros y se usa la mejor para predecir, para poder finalmente contrastar su desempeño contra el mejor modelo que se obtuvo para esta serie en el laboratorio anterior.

Cabe mencionar que al ser una serie agregada tiende a ser más estable que las series individuales, ya que el ruido de cada punto de ingreso se compensa al sumarlos, de tal forma que en principio debería ser un escenario más favorable para el modelo. No obstante, arrastra el mismo problema que todas las demás, y es que el entrenamiento termina justo en el fondo del desplome de la pandemia mientras que el período de prueba es la recuperación, lo cual es un contexto bastante exigente para cualquier modelo.

## **Conjuntos de entrenamiento y prueba**

Para poder comparar los modelos LSTM contra los que ya se construyeron, se trabaja con **los mismos
conjuntos de entrenamiento y prueba del Laboratorio 1**, sin volver a decidir nada sobre los datos.
Es decir, se parte de los mismos `entrenamiento.csv` y `prueba.csv`, que ya vienen filtrados a
Turista + Excursionista y particionados de forma temporal 70/30, de tal forma que el entrenamiento
cubre de enero de 2009 a marzo de 2021 (147 meses) y la prueba de abril de 2021 a junio de 2026
(63 meses). Cabe mencionar que la partición respeta el orden cronológico y no es aleatoria, ya que
en una serie de tiempo ese orden es justamente lo que se quiere modelar.

Bajo esta idea, la serie de **total mensual de viajeros** se reconstruye sumando los viajeros de todas las regiones, vías y fronteras por mes, y se le aplica la misma
transformación logarítmica que se justificó en el laboratorio anterior, dado que el lambda de
Box-Cox sobre el tramo sin pandemia (2009-2019) resultó en -0.318, es decir cercano a 0. Adicional,
las métricas se calculan igual que antes, deshaciendo el logaritmo con la exponencial para reportar
el MAE y el RMSE en número de viajeros, de tal forma que los resultados sean comparables uno a uno
contra el SARIMA, Holt-Winters, el suavizamiento exponencial simple, el seasonal naive y Prophet.


In [1]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# El codigo compartido vive en src/ (construccion de las series, metricas y utilidades de LSTM).
RAIZ = Path.cwd().parents[1] if Path.cwd().name == "lstm" else Path.cwd()
sys.path.insert(0, str(RAIZ / "src"))

import config
import utils

warnings.filterwarnings("ignore")
pd.set_option("display.width", 120)
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (13, 5)

SERIE = "total"
PERIODO = config.PERIODO

# Las series se reconstruyen desde los CSV del Laboratorio 1, con la misma agrupacion mensual.
serie_train = utils.construir_serie(SERIE, "train")
serie_prueba = utils.construir_serie(SERIE, "prueba")

# Transformacion logaritmica, la misma que se eligio en el Laboratorio 1 para esta serie.
serie_train_log = np.log(serie_train)
serie_prueba_log = np.log(serie_prueba)

print("Serie:", config.SERIES_LAB1[SERIE]["etiqueta"])
print(f"Entrenamiento: {serie_train.index.min():%Y-%m} a {serie_train.index.max():%Y-%m}  "
      f"({len(serie_train)} meses)")
print(f"Prueba       : {serie_prueba.index.min():%Y-%m} a {serie_prueba.index.max():%Y-%m}  "
      f"({len(serie_prueba)} meses)")
print("Frecuencia   :", serie_train.index.freqstr, "(mensual, inicio de mes)")
print("Meses sin registro:", int(serie_train.isna().sum() + serie_prueba.isna().sum()))
print()
print("Mejor modelo del Laboratorio 1 para esta serie:")
print(f"  {config.MEJORES_LAB1[SERIE]['modelo']}  "
      f"MAE={config.MEJORES_LAB1[SERIE]['MAE']:,.0f}  "
      f"RMSE={config.MEJORES_LAB1[SERIE]['RMSE']:,.0f}")

serie_train.tail()


Serie: Total mensual
Entrenamiento: 2009-01 a 2021-03  (147 meses)
Prueba       : 2021-04 a 2026-06  (63 meses)
Frecuencia   : MS (mensual, inicio de mes)
Meses sin registro: 0

Mejor modelo del Laboratorio 1 para esta serie:
  Suav. exp. simple  MAE=158,777  RMSE=173,709


Fecha
2020-11-01    52865.000000
2020-12-01    55620.000000
2021-01-01    89960.000000
2021-02-01    52605.000000
2021-03-01    85017.396333
Freq: MS, Name: Viajero, dtype: float64